# LeetCode 2303 — Calculate Amount Paid in Taxes bằng Python

Link đề: https://leetcode.com/problems/calculate-amount-paid-in-taxes/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Math

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **thuế lũy tiến theo bậc (progressive tax brackets)**: mỗi đoạn thu nhập chịu một mức thuế riêng, không phải cả thu nhập chịu một mức.
2. Biết đọc mảng 2 chiều `brackets[i] = [upper_i, percent_i]` và tính **độ rộng của từng bậc** `upper_i - upper_{i-1}`.
3. Biết dùng `min`, `max` để cắt đoạn thu nhập vào từng bậc, và hiểu vì sao có thể **dừng sớm** khi hết thu nhập.
4. Viết 5 cách: duyệt bậc với mốc trước, trừ dần thu nhập còn lại, dùng `min(upper, income)`, tính tổng số nguyên rồi chia `100` (tránh sai số thực), và cách tích lũy bằng `zip`/hàm sinh.
5. Trace, test ngẫu nhiên đối chiếu (so sánh số thực bằng `math.isclose`), và tránh lỗi: áp thuế suất cho cả thu nhập, quên trừ mốc trước, dùng chia nguyên.

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `brackets`, trong đó `brackets[i] = [upper_i, percent_i]` nghĩa là bậc thuế thứ `i` có **cận trên** là `upper_i` và thuế suất là `percent_i` phần trăm. Các bậc được sắp theo `upper_i` **tăng dần ngặt**.

Thuế tính như sau:

- Phần thu nhập từ `0` đến `upper_0` chịu thuế `percent_0 %`.
- Phần thu nhập từ `upper_0` (không tính) đến `upper_1` chịu thuế `percent_1 %`.
- Phần thu nhập từ `upper_1` đến `upper_2` chịu thuế `percent_2 %`, ... và cứ thế.

Cho `income` (thu nhập), trả về **tổng thuế phải nộp** dưới dạng số thực. Kết quả chấp nhận sai số tới `10^-5`.

| Ví dụ | brackets | income | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[[3,50],[7,10],[12,25]]` | `10` | `2.65` | `3*50% + 4*10% + 3*25% = 1.5 + 0.4 + 0.75` |
| 2 | `[[1,0],[4,25],[5,50]]` | `2` | `0.25` | `1*0% + 1*25%` |
| 3 | `[[2,50]]` | `0` | `0.0` | không có thu nhập nên không có thuế |

Constraints: `1 <= brackets.length <= 100`; `1 <= upper_i <= 1000`; `0 <= percent_i <= 100`; `0 <= income <= 1000`; `upper_i` tăng dần ngặt; `upper` của bậc cuối `>= income` (thu nhập luôn nằm trong các bậc).

In [1]:
examples = [
    ([[3, 50], [7, 10], [12, 25]], 10, 2.65),
    ([[1, 0], [4, 25], [5, 50]], 2, 0.25),
    ([[2, 50]], 0, 0.0),
]
for brackets, income, expected in examples:
    prev, total, parts = 0, 0.0, []
    for upper, pct in brackets:
        taxable = max(0, min(upper, income) - prev)
        parts.append((prev, upper, taxable, pct))
        total += taxable * pct / 100
        prev = upper
    print(f"income={income}: (từ, đến, phần chịu thuế, %)={parts} -> tổng={total:.2f} (mong đợi {expected})")

income=10: (từ, đến, phần chịu thuế, %)=[(0, 3, 3, 50), (3, 7, 4, 10), (7, 12, 3, 25)] -> tổng=2.65 (mong đợi 2.65)
income=2: (từ, đến, phần chịu thuế, %)=[(0, 1, 1, 0), (1, 4, 1, 25), (4, 5, 0, 50)] -> tổng=0.25 (mong đợi 0.25)
income=0: (từ, đến, phần chịu thuế, %)=[(0, 2, 0, 50)] -> tổng=0.00 (mong đợi 0.0)


## 3. Kiến thức nền

### 3.1. Thuế lũy tiến: mỗi đoạn một mức

Không phải "thu nhập 10 thì áp thuế suất của bậc chứa 10 cho cả 10". Mỗi **đoạn** của thu nhập chịu mức riêng: đoạn `(0, 3]` chịu `50%`, đoạn `(3, 7]` chịu `10%`, đoạn `(7, 12]` chịu `25%`. Thu nhập `10` chạm cả ba đoạn; phần chạm đoạn cuối chỉ là `10 - 7 = 3`.

In [2]:
brackets = [[3, 50], [7, 10], [12, 25]]
prev = 0
for upper, pct in brackets:
    print(f"đoạn ({prev}, {upper}] rộng {upper - prev}, thuế suất {pct}%")
    prev = upper

đoạn (0, 3] rộng 3, thuế suất 50%
đoạn (3, 7] rộng 4, thuế suất 10%
đoạn (7, 12] rộng 5, thuế suất 25%


### 3.2. Giải nén cặp bằng `for upper, pct in brackets`

Mỗi phần tử của `brackets` là list 2 số. `for upper, pct in brackets` **giải nén (unpacking)** thẳng thành hai biến; gọn hơn `bracket[0]`, `bracket[1]`.

In [3]:
brackets = [[3, 50], [7, 10], [12, 25]]
for upper, pct in brackets:
    print(upper, pct)
first, second = brackets[0]
print(first, second)

3 50
7 10
12 25
3 50


### 3.3. `min` và `max` để "cắt" đoạn

Phần thu nhập rơi vào đoạn `(prev, upper]` là `max(0, min(income, upper) - prev)`:

- `min(income, upper)` không cho vượt cận trên của bậc.
- Trừ `prev` được độ rộng thực dùng; nếu thu nhập chưa tới `prev` thì âm, nên `max(0, ...)` về `0`.

In [4]:
income = 10
for prev, upper in ((0, 3), (3, 7), (7, 12), (12, 20)):
    print((prev, upper), max(0, min(income, upper) - prev))

(0, 3) 3
(3, 7) 4
(7, 12) 3
(12, 20) 0


### 3.4. Số thực và sai số: `math.isclose`

Cộng nhiều số thập phân dễ ra sai số (`0.1 + 0.2 != 0.3`). LeetCode cho sai số `1e-5`; trong test của ta so sánh bằng `math.isclose(a, b, abs_tol=1e-5)`. Để tránh sai số tận gốc, có thể cộng **số nguyên** `taxable * percent` rồi chia `100` một lần ở cuối.

In [5]:
import math

print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(math.isclose(0.1 + 0.2, 0.3, abs_tol=1e-5))
print(15 + 4 * 10 + 3 * 25, (15 * 10 + 4 * 10 + 3 * 25) / 100)

False 0.30000000000000004
True
130 2.65


### 3.5. Phép chia: `/` (thực) và `//` (nguyên)

`265 / 100 = 2.65`, còn `265 // 100 = 2`. Chia nguyên làm mất phần thập phân và cho sai đáp án.

In [6]:
print(265 / 100, 265 // 100)
print(7 / 2, 7 // 2)

2.65 2
3.5 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 (`brackets=[[3,50],[7,10],[12,25]]`, `income=10`):

1. Bậc 1: phần `0..3` rộng `3`, thuế `3 * 50% = 1.5`. Thu nhập `10` vượt cận `3`, nên đủ cả phần.
2. Bậc 2: phần `3..7` rộng `4`, thuế `4 * 10% = 0.4`. `10` vượt `7`, nên đủ.
3. Bậc 3: phần `7..12` nhưng thu nhập chỉ tới `10`, nên phần chịu thuế là `10 - 7 = 3`, thuế `3 * 25% = 0.75`.
4. Cộng: `1.5 + 0.4 + 0.75 = 2.65`.

Mô hình trực giác: **đổ nước vào các bể nối tiếp**. Mỗi bể có sức chứa là độ rộng bậc; nước (thu nhập) đổ vào bể 1 cho đầy, rồi tràn sang bể 2, ... Lượng nước ở mỗi bể nhân với thuế suất của bể đó.

In [7]:
income = 10
rest = income
tax = 0.0
prev = 0
for upper, pct in [[3, 50], [7, 10], [12, 25]]:
    width = upper - prev
    used = min(rest, width)
    tax += used * pct / 100
    print(f"bể rộng {width}: đổ {used}, còn {rest - used}, cộng thuế {used * pct / 100}")
    rest -= used
    prev = upper
print("tổng thuế:", tax)

bể rộng 3: đổ 3, còn 7, cộng thuế 1.5
bể rộng 4: đổ 4, còn 3, cộng thuế 0.4
bể rộng 5: đổ 3, còn 0, cộng thuế 0.75
tổng thuế: 2.65


## 5. Cách 1 — Brute Force / baseline: duyệt bậc với mốc trước, cắt bằng `min`/`max`

**Ý tưởng:** giữ `prev` là cận trên của bậc trước. Với mỗi bậc `[upper, pct]`, phần chịu thuế là `max(0, min(income, upper) - prev)`; cộng `phần * pct / 100`; cập nhật `prev = upper`.

**Độ phức tạp:** duyệt một lần `O(b)` với `b = len(brackets) <= 100`. Cách này đã tối ưu về bậc; "nút thắt" duy nhất là sai số số thực, ta cải thiện ở Cách 4.

In [8]:
from typing import List

class SolutionPrev:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        tax, prev = 0.0, 0
        for upper, pct in brackets:
            taxable = max(0, min(income, upper) - prev)
            tax += taxable * pct / 100
            prev = upper
        return tax

for b, inc in (([[3, 50], [7, 10], [12, 25]], 10), ([[1, 0], [4, 25], [5, 50]], 2), ([[2, 50]], 0)):
    print(b, inc, "->", SolutionPrev().calculateTax(b, inc))

[[3, 50], [7, 10], [12, 25]] 10 -> 2.65
[[1, 0], [4, 25], [5, 50]] 2 -> 0.25
[[2, 50]] 0 -> 0.0


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (dừng sớm):** các bậc sắp tăng dần. Khi `income <= prev` thì mọi bậc sau đều không có phần nào chịu thuế, nên có thể `break` ngay.

**Insight 2 (số nguyên):** thay vì cộng số thực, tổng `sum(taxable * pct)` là số **nguyên**; chia `100` một lần ở cuối cho ra kết quả chính xác nhất có thể.

**Invariant:** sau khi xử lý `k` bậc đầu, `tax` bằng tổng thuế của phần thu nhập nằm trong `(0, upper_{k-1}]`.

**Quy tắc:** với mỗi bậc, `taxable = min(income, upper) - prev` (không âm nếu chưa break); `tax += taxable * pct`; `prev = upper`; nếu `income <= upper` thì dừng.

## 7. Các cách tiếp theo

### Cách 2 — Trừ dần thu nhập còn lại (mô hình đổ nước)

Giữ `rest = income`. Mỗi bậc rộng `width = upper - prev`, lấy `used = min(rest, width)`, trừ `rest -= used`, dừng khi `rest == 0`.

In [9]:
from typing import List

class SolutionRest:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        rest, prev, tax = income, 0, 0.0
        for upper, pct in brackets:
            if rest == 0:
                break
            used = min(rest, upper - prev)
            tax += used * pct / 100
            rest -= used
            prev = upper
        return tax

for b, inc in (([[3, 50], [7, 10], [12, 25]], 10), ([[1, 0], [4, 25], [5, 50]], 2), ([[2, 50]], 0)):
    print(b, inc, "->", SolutionRest().calculateTax(b, inc))

[[3, 50], [7, 10], [12, 25]] 10 -> 2.65
[[1, 0], [4, 25], [5, 50]] 2 -> 0.25
[[2, 50]] 0 -> 0.0


**Độ phức tạp Cách 2:** Time `O(b)` (dừng sớm khi `rest == 0`), Space `O(1)`.

### Cách 3 — Dừng sớm bằng `break` khi đã qua thu nhập

Nếu bậc hiện tại có `upper >= income`, đây là bậc cuối cần tính: phần chịu thuế là `income - prev`, cộng xong thì `break`. Không cần `min`/`max`.

In [10]:
from typing import List

class SolutionBreak:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        tax, prev = 0.0, 0
        for upper, pct in brackets:
            if income <= prev:
                break
            if income >= upper:
                tax += (upper - prev) * pct / 100       # bậc được lấp đầy
            else:
                tax += (income - prev) * pct / 100      # bậc cuối, lấp một phần
                break
            prev = upper
        return tax

for b, inc in (([[3, 50], [7, 10], [12, 25]], 10), ([[1, 0], [4, 25], [5, 50]], 2), ([[2, 50]], 0)):
    print(b, inc, "->", SolutionBreak().calculateTax(b, inc))

[[3, 50], [7, 10], [12, 25]] 10 -> 2.65
[[1, 0], [4, 25], [5, 50]] 2 -> 0.25
[[2, 50]] 0 -> 0.0


**Độ phức tạp Cách 3:** Time `O(b)` tốt nhất khi dừng sớm, Space `O(1)`.

### Cách 4 — Cộng số nguyên rồi chia `100` một lần

Tránh sai số thực: `total = sum(taxable * pct)` là số nguyên (các thành phần đều nguyên), rồi `total / 100`.

In [11]:
from typing import List

class SolutionInteger:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        total, prev = 0, 0
        for upper, pct in brackets:
            total += max(0, min(income, upper) - prev) * pct
            prev = upper
        return total / 100

for b, inc in (([[3, 50], [7, 10], [12, 25]], 10), ([[1, 0], [4, 25], [5, 50]], 2), ([[2, 50]], 0)):
    print(b, inc, "->", SolutionInteger().calculateTax(b, inc))

[[3, 50], [7, 10], [12, 25]] 10 -> 2.65
[[1, 0], [4, 25], [5, 50]] 2 -> 0.25
[[2, 50]] 0 -> 0.0


**Độ phức tạp Cách 4:** Time `O(b)`, Space `O(1)`. Chỉ có **một** phép chia thực, nên kết quả chính xác nhất có thể biểu diễn.

### Cách 5 — Dựng danh sách các mốc rồi `zip`

Tạo danh sách mốc trước `prevs = [0] + [upper...]` rồi `zip(prevs, brackets)` ghép mỗi bậc với mốc trước của nó. Biểu thức tổng dạng `sum(...)` một dòng.

In [12]:
from typing import List

class SolutionZip:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        prevs = [0] + [upper for upper, _ in brackets]
        total = sum(max(0, min(income, upper) - prev) * pct
                    for prev, (upper, pct) in zip(prevs, brackets))
        return total / 100

for b, inc in (([[3, 50], [7, 10], [12, 25]], 10), ([[1, 0], [4, 25], [5, 50]], 2), ([[2, 50]], 0)):
    print(b, inc, "->", SolutionZip().calculateTax(b, inc))

[[3, 50], [7, 10], [12, 25]] 10 -> 2.65
[[1, 0], [4, 25], [5, 50]] 2 -> 0.25
[[2, 50]] 0 -> 0.0


**Độ phức tạp Cách 5:** Time `O(b)`, Space `O(b)` cho danh sách `prevs`.

## 8. TRACE từng bước

Trace Cách 1 với `brackets = [[3,50],[7,10],[12,25]]`, `income = 10`:

| Bậc | `prev` | `upper` | `min(income, upper)` | `taxable` | `pct` | cộng thuế | `tax` |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 3 | 3 | 3 | 50 | 1.5 | 1.5 |
| 2 | 3 | 7 | 7 | 4 | 10 | 0.4 | 1.9 |
| 3 | 7 | 12 | 10 | 3 | 25 | 0.75 | 2.65 |

Kết quả `2.65`.

In [13]:
def trace_prev(brackets, income):
    tax, prev = 0.0, 0
    print(f"income={income}")
    for k, (upper, pct) in enumerate(brackets, 1):
        cap = min(income, upper)
        taxable = max(0, cap - prev)
        add = taxable * pct / 100
        tax += add
        print(f"  bậc {k}: prev={prev} upper={upper} min(income,upper)={cap} taxable={taxable} pct={pct} +{add} -> tax={tax}")
        prev = upper
    return tax

print("kết quả:", trace_prev([[3, 50], [7, 10], [12, 25]], 10))
print("kết quả:", trace_prev([[1, 0], [4, 25], [5, 50]], 2))

income=10
  bậc 1: prev=0 upper=3 min(income,upper)=3 taxable=3 pct=50 +1.5 -> tax=1.5
  bậc 2: prev=3 upper=7 min(income,upper)=7 taxable=4 pct=10 +0.4 -> tax=1.9
  bậc 3: prev=7 upper=12 min(income,upper)=10 taxable=3 pct=25 +0.75 -> tax=2.65
kết quả: 2.65
income=2
  bậc 1: prev=0 upper=1 min(income,upper)=1 taxable=1 pct=0 +0.0 -> tax=0.0
  bậc 2: prev=1 upper=4 min(income,upper)=2 taxable=1 pct=25 +0.25 -> tax=0.25
  bậc 3: prev=4 upper=5 min(income,upper)=2 taxable=0 pct=50 +0.0 -> tax=0.25
kết quả: 0.25


In [14]:
def trace_rest(brackets, income):
    rest, prev, tax = income, 0, 0.0
    for k, (upper, pct) in enumerate(brackets, 1):
        if rest == 0:
            print(f"  bậc {k}: rest=0 -> dừng sớm")
            break
        used = min(rest, upper - prev)
        tax += used * pct / 100
        rest -= used
        print(f"  bậc {k}: bể rộng {upper - prev}, đổ {used}, rest={rest}, tax={tax}")
        prev = upper
    return tax

print("kết quả:", trace_rest([[1, 0], [4, 25], [5, 50]], 2))

  bậc 1: bể rộng 1, đổ 1, rest=1, tax=0.0
  bậc 2: bể rộng 3, đổ 1, rest=0, tax=0.25
  bậc 3: rest=0 -> dừng sớm
kết quả: 0.25


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        total, prev = 0, 0
        for upper, pct in brackets:
            total += max(0, min(income, upper) - prev) * pct
            prev = upper
        return total / 100

print(Solution().calculateTax([[3, 50], [7, 10], [12, 25]], 10))
print(Solution().calculateTax([[1, 0], [4, 25], [5, 50]], 2))
print(Solution().calculateTax([[2, 50]], 0))

2.65
0.25
0.0


## 10. Vì sao thuật toán đúng

- **Phân hoạch thu nhập.** Các đoạn `(prev, upper]` rời nhau và phủ `(0, upper_cuối]`. Thu nhập `income` nằm trong phạm vi đó (đề đảm bảo `upper_cuối >= income`), nên mỗi đơn vị thu nhập thuộc **đúng một** đoạn.
- **Cắt đoạn đúng.** Số đơn vị thu nhập thuộc đoạn `(prev, upper]` là `|{x : prev < x <= min(income, upper)}| = max(0, min(income, upper) - prev)`.
- **Tổng theo từng đoạn.** Mỗi đơn vị trong đoạn đó chịu `pct/100`, nên thuế của đoạn là `taxable * pct / 100`; tổng các đoạn là thuế toàn phần.
- **Bậc sau thu nhập đóng góp 0.** Nếu `income <= prev` thì `min(income, upper) - prev <= 0`, nên `max(0, ...) = 0`; điều đó cho phép dừng sớm mà không đổi kết quả.

## 11. Phân tích Time / Space Complexity

Gọi `b = len(brackets)` (`<= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Mốc trước + `min`/`max` | `O(b)` | `O(1)` |
| 2. Trừ dần `rest` | `O(b)` (dừng sớm) | `O(1)` |
| 3. `break` khi qua thu nhập | `O(b)` (dừng sớm) | `O(1)` |
| 4. Số nguyên rồi chia `100` | `O(b)` | `O(1)` |
| 5. `zip` + `sum` | `O(b)` | `O(b)` |

Dữ liệu rất nhỏ (`b <= 100`), mọi cách đều tức thì. Khác biệt đáng kể nhất là **độ chính xác số thực**, không phải tốc độ.

## 12. Bộ test

Đối chiếu tất cả các cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed` (so sánh số thực bằng `math.isclose`).

In [16]:
import math, random

solutions = [SolutionPrev(), SolutionRest(), SolutionBreak(), SolutionInteger(), SolutionZip(), Solution()]

def close(a, b):
    return math.isclose(a, b, abs_tol=1e-5)

fixed = [
    ([[3, 50], [7, 10], [12, 25]], 10, 2.65),
    ([[1, 0], [4, 25], [5, 50]], 2, 0.25),
    ([[2, 50]], 0, 0.0),
    ([[10, 100]], 10, 10.0),                    # thuế suất 100%
    ([[5, 0], [10, 50]], 5, 0.0),               # thu nhập đúng bằng cận trên bậc đầu
    ([[5, 10], [10, 20]], 7, 0.5 + 0.4),        # thu nhập vào giữa bậc 2
    ([[1000, 100]], 1000, 1000.0),              # giá trị lớn nhất
    ([[1, 7], [2, 7], [3, 7]], 3, 0.21),        # cùng thuế suất
]
for b, inc, expected in fixed:
    for s in solutions:
        got = s.calculateTax(b, inc)
        assert close(got, expected), (type(s).__name__, b, inc, got, expected)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 8 test cố định cho 6 cách


In [17]:
random.seed(2303)
for _ in range(1000):
    b = random.randint(1, 8)
    uppers = sorted(random.sample(range(1, 1001), b))
    brackets = [[u, random.randint(0, 100)] for u in uppers]
    income = random.randint(0, uppers[-1])
    ref = SolutionPrev().calculateTax(brackets, income)
    for s in solutions[1:]:
        assert close(s.calculateTax(brackets, income), ref), (type(s).__name__, brackets, income)
print("1000 test ngẫu nhiên đối chiếu: OK")

1000 test ngẫu nhiên đối chiếu: OK


In [18]:
# so với định nghĩa trực tiếp: đếm từng đơn vị thu nhập (O(income * b)) làm "chân lý"
def truth(brackets, income):
    total = 0
    for x in range(1, income + 1):
        prev = 0
        for upper, pct in brackets:
            if x <= upper:
                total += pct
                break
            prev = upper
    return total / 100

random.seed(7)
for _ in range(300):
    b = random.randint(1, 6)
    uppers = sorted(random.sample(range(1, 60), b))
    brackets = [[u, random.randint(0, 100)] for u in uppers]
    income = random.randint(0, uppers[-1])
    assert close(Solution().calculateTax(brackets, income), truth(brackets, income)), (brackets, income)
print("khớp định nghĩa đếm từng đơn vị: OK")

khớp định nghĩa đếm từng đơn vị: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Áp thuế suất của bậc cuối cho **cả** thu nhập

Đây là thuế **phẳng**, không phải lũy tiến. Ví dụ 1 sẽ ra `10 * 25% = 2.5` thay vì `2.65`.

In [19]:
def sai_thue_phang(brackets, income):
    for upper, pct in brackets:
        if income <= upper:
            return income * pct / 100              # SAI

print("sai :", sai_thue_phang([[3, 50], [7, 10], [12, 25]], 10))
print("đúng:", Solution().calculateTax([[3, 50], [7, 10], [12, 25]], 10))

sai : 2.5
đúng: 2.65


### Lỗi 2: Quên trừ mốc trước (`prev`)

Dùng `min(income, upper) * pct` cho từng bậc làm đếm lặp: thu nhập vùng bậc trước bị tính lại ở bậc sau.

In [20]:
def sai_quen_prev(brackets, income):
    total = 0
    for upper, pct in brackets:
        total += min(income, upper) * pct          # SAI: thiếu - prev
    return total / 100

print("sai :", sai_quen_prev([[3, 50], [7, 10], [12, 25]], 10))
print("đúng:", Solution().calculateTax([[3, 50], [7, 10], [12, 25]], 10))

sai : 4.7
đúng: 2.65


### Lỗi 3: Không chặn phần âm khi thu nhập thấp

Nếu thu nhập `2` mà ở bậc có `prev = 3`, `min(income, upper) - prev = -1`. Không `max(0, ...)` thì cộng thuế âm.

In [21]:
def sai_khong_max0(brackets, income):
    total, prev = 0, 0
    for upper, pct in brackets:
        total += (min(income, upper) - prev) * pct   # SAI: có thể âm
        prev = upper
    return total / 100

print("sai :", sai_khong_max0([[3, 50], [7, 10], [12, 25]], 2))
print("đúng:", Solution().calculateTax([[3, 50], [7, 10], [12, 25]], 2))

sai : -0.35
đúng: 1.0


### Lỗi 4: Chia nguyên `//`

`taxable * pct // 100` bỏ phần thập phân (vd `3 * 50 // 100 = 1` thay vì `1.5`).

In [22]:
print(3 * 50 // 100, 3 * 50 / 100)
def sai_chia_nguyen(brackets, income):
    total, prev = 0, 0
    for upper, pct in brackets:
        total += max(0, min(income, upper) - prev) * pct // 100   # SAI: cắt cụt từng phần
        prev = upper
    return total

print("sai :", sai_chia_nguyen([[3, 50], [7, 10], [12, 25]], 10), "| đúng:", Solution().calculateTax([[3, 50], [7, 10], [12, 25]], 10))

1 1.5
sai : 1 | đúng: 2.65


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "bậc", "khoảng", "từng đoạn chịu mức riêng" là nhớ ngay: **cắt đoạn** bằng `min`/`max` và dùng **mốc trước** (`prev`).
- Dữ liệu sắp tăng dần -> có thể **dừng sớm**.
- Tránh sai số thực: cộng số nguyên rồi chia `100` một lần.
- Kiểm tra: ví dụ 1 (`2.65`), `income = 0` (`0.0`), `income` đúng bằng một cận.
- Pattern tương tự: bảng giá điện/nước theo bậc, tiền thuê taxi theo km, cước vận chuyển.

## 15. Biến thể của bài

- **Bậc cận trên không có hạn** (`upper = infinity` cho bậc cuối) — dùng `float('inf')`.
- **Tính thu nhập ròng** (`income - tax`) hoặc **thuế suất trung bình** `tax / income`.
- **Cho thuế nộp, tìm thu nhập** (hàm ngược: tìm `income` có `tax(income) = T`; hàm đơn điệu nên **tìm nhị phân**).
- **Mức giảm trừ/khấu trừ (deduction)** trừ vào thu nhập trước khi tính.
- **Nhiều người, tính tổng thuế** bằng cách dùng **mảng cộng dồn (prefix sum)** các mức thuế để trả lời truy vấn `O(log b)` với `bisect`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mốc trước + `min`/`max` | cắt đoạn từng bậc | `O(b)` | `O(1)` | cách chuẩn, dễ nghĩ |
| 2. Trừ dần `rest` | đổ nước vào bể | `O(b)` | `O(1)` | trực quan, dừng sớm |
| 3. `break` sớm | bậc đầy / bậc cuối | `O(b)` | `O(1)` | tránh `min`/`max` |
| 4. Số nguyên chia `100` | cộng nguyên, chia một lần | `O(b)` | `O(1)` | nộp bài, chính xác nhất |
| 5. `zip` + `sum` | một biểu thức tổng | `O(b)` | `O(b)` | code ngắn, hàm thuần |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().calculateTax([[5, 10], [10, 20]], 5)`.

**Bài 2 (tính tay).** Với `brackets = [[4, 25], [9, 40], [20, 60]]`, `income = 12`, tính từng đoạn rồi tổng thuế.

**Bài 3 (điền TODO).** Hoàn thành `taxable_in_bracket(prev, upper, income)` trả về phần thu nhập rơi vào đoạn `(prev, upper]` (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `net_income(brackets, income)` trả về thu nhập sau thuế.

**Bài 5 (mini challenge).** Viết `income_for_tax(brackets, tax)` tìm thu nhập **nguyên** nhỏ nhất có thuế `>= tax` bằng tìm nhị phân (hàm thuế đơn điệu không giảm theo `income`); nếu không đạt trả `-1`.

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def taxable_in_bracket_todo(prev, upper, income):
    # return max(0, min(...) - ...)
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: thu nhập `5` chỉ chạm bậc 1 (`0..5`), thuế `5 * 10% = 0.5`. Bài 2: các đoạn `(0,4]`, `(4,9]`, `(9,12]` có phần chịu thuế `4, 5, 3`.
- **Mức 2 (vừa):** Bài 3: `max(0, min(income, upper) - prev)`. Bài 4: `income - Solution().calculateTax(...)`.
- **Mức 3 (sâu):** Bài 5: `lo, hi = 0, brackets[-1][0]`; nếu thuế tại `hi` `< tax` thì `-1`; tìm nhị phân như "first true". Cẩn thận so sánh số thực: dùng số nguyên (nhân `100`) cho chắc.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
import math

# Bài 1 và 2
assert math.isclose(Solution().calculateTax([[5, 10], [10, 20]], 5), 0.5)
assert math.isclose(Solution().calculateTax([[4, 25], [9, 40], [20, 60]], 12), 4 * 0.25 + 5 * 0.40 + 3 * 0.60)

# Bài 3
def taxable_in_bracket(prev, upper, income):
    return max(0, min(income, upper) - prev)

# Bài 4
def net_income(brackets, income):
    return income - Solution().calculateTax(brackets, income)

# Bài 5
def income_for_tax(brackets, tax):
    lo, hi = 0, brackets[-1][0]
    if Solution().calculateTax(brackets, hi) < tax - 1e-9:
        return -1
    while lo < hi:
        mid = (lo + hi) // 2
        if Solution().calculateTax(brackets, mid) >= tax - 1e-9:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert taxable_in_bracket(3, 7, 10) == 4 and taxable_in_bracket(7, 12, 10) == 3 and taxable_in_bracket(7, 12, 5) == 0
assert math.isclose(net_income([[3, 50], [7, 10], [12, 25]], 10), 7.35)
b = [[3, 50], [7, 10], [12, 25]]
assert income_for_tax(b, 1.5) == 3
assert income_for_tax(b, 2.65) == 10
assert income_for_tax(b, 100) == -1
assert income_for_tax(b, 0) == 0
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cắt đoạn bằng mốc trước, cộng số nguyên, chia 100 một lần)

```python
class Solution:
    def calculateTax(self, brackets: List[List[int]], income: int) -> float:
        total, prev = 0, 0
        for upper, pct in brackets:
            total += max(0, min(income, upper) - prev) * pct
            prev = upper
        return total / 100
```

**Checklist**

- [ ] Hiểu đây là thuế **lũy tiến**: mỗi đoạn một mức, không áp một mức cho cả thu nhập?
- [ ] Có trừ mốc trước `prev` và cập nhật `prev = upper` sau mỗi bậc?
- [ ] Có chặn phần âm bằng `max(0, ...)` hoặc `break` khi qua thu nhập?
- [ ] Dùng `/ 100` (chia thực), không phải `// 100`?
- [ ] Đã thử `income = 0`, `income` đúng bằng một cận trên, và bậc có thuế suất `0` hoặc `100`?
- [ ] Signature đúng `calculateTax(self, brackets: List[List[int]], income: int) -> float` và có `from typing import List`?